# Regression Analysis
## Importing libraries and creating a NaN filtered dataset

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pandas as pd

penguins = sns.load_dataset("penguins").dropna() # dopna removes rows with NaN data points

penguins.describe()


## Check models with variables of each non-categorical variable

In [ ]:

variables = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
variables2 = ["species", "sex"]
xLen = max(len(variables), len(variables2))
fig, axes = plt.subplots(2, xLen, figsize=(15,8))
name = "CDV"


sns.set()

for i, var in enumerate(variables):
    model = smf.ols("body_mass_g ~" + var, data=penguins)
    result = model.fit()
    print(f"{name} {var}'s R²: {result.rsquared}")
    sns.scatterplot(
        data=penguins,
        x=var,
        y="body_mass_g",
        ax=axes[0][i],
        hue="sex",
        palette = {
          "Male": "tab:blue",
          "Female": "tab:pink",
        },
        style="species",
        legend=False
    )

for i, var in enumerate(variables2):
    model = smf.ols("body_mass_g ~" + var, data=penguins)
    result = model.fit()
    print(f"{name} {var}'s R²: {result.rsquared}")
    sns.boxplot(
        data=penguins,
        x=var,
        y="body_mass_g",
        ax=axes[1][i],
        legend=False
    )

sns.scatterplot(
    data=penguins,
    x=variables[-1],
    y="body_mass_g",
    ax=axes[0][-1],
    hue="sex",
    palette = {
      "Male": "tab:blue",
      "Female": "tab:pink",
    },
    style="species",
    legend="full"
)
handles, labels = axes[0][-1].get_legend_handles_labels()
axes[0][-1].legend().remove()
fig.legend(handles, labels, bbox_to_anchor=(1.0,0.5), loc="center left", borderaxespad=0)
fig.tight_layout()
fig.savefig(name + ".jpg")

plt.show()

billFit = smf.ols("body_mass_g ~ bill_depth_mm", data=penguins).fit()
print(f"\n\nBill depth slope: {billFit.params['bill_depth_mm']}")
billFit = smf.ols("body_mass_g ~ bill_depth_mm", data=penguins[penguins["species"] != "Gentoo"]).fit()
print(f"Bill depth slope no gentoo: {billFit.params['bill_depth_mm']}")
billFit = smf.ols("body_mass_g ~ bill_depth_mm", data=penguins[penguins["species"] == "Gentoo"]).fit()
print(f"Bill depth slope only gentoo: {billFit.params['bill_depth_mm']}")

speciesBillDepthEffect = smf.ols("bill_depth_mm ~ species", penguins).fit()
print(f"\n\nBill depth for gentoo: {speciesBillDepthEffect.params['species[T.Gentoo]']}")
print(f"Bill depth for chinstrap: {speciesBillDepthEffect.params['species[T.Chinstrap]']}")


Flipper length seems to be the best variable for body mass so far, as the R² value is closest to 1.
It can also be seen that both bill length and depth varies widely based on the species of penguin, which might be the reason for their low R² values. One can also see in the plot that the reason flipper length has a comparatively high R² value is that longer flipper length correlates with species and sex with higher body mass, which could be the real variables.
Notably, the bill depth has an inverse correlation with body mass when including all species. This is because there's a positive correlation within 2 different groupings in the plots (the gentoo penguins and the pooled adelie + chinstrap penguins), but the adelie and chinstrap penguins seem to have deeper bills than gentoo penguins despite their lower body mass (gentoo having aboout -3mm compared to both adelie and chinstrap penguins), affecting the correlation. This is known as the simpson's paradox, where a slope present in multiple subgroups can disappear or invert when including all groups in one model.
The estimated formula for body_mass_g ~ flipper_length_mm are:
$\hat{y} = 50.15*x - 5872.09$
In other words, $\hat{\beta}_1 = 50.1533$ and $\hat{\beta}_0 = -5872.0927$

## Filtering by species and sex

In [ ]:

rows = []
for (sp, sx), g in penguins.groupby(["species", "sex"]):
  fit = smf.ols("body_mass_g ~ flipper_length_mm", data=g).fit()
  rows.append({"species": sp, "sex": sx, "n": len(g), "r": g["flipper_length_mm"].corr(g["body_mass_g"]), "slope": fit.params["flipper_length_mm"], "R2": fit.rsquared})
print(pd.DataFrame(rows).round(2))


Here we can see that within the categories of sexes and species, the slopes are way lower than the 50.1533 estimate of our model. This is consistent with the theory that flipper_length is a useful variable mostly due to differences between species.

## More useful plots
### Linear plot

In [ ]:

filename = "linearRegression"
fig, axes = plt.subplots(1,1, figsize=(5,5))
sns.regplot(x=penguins["flipper_length_mm"], y=penguins["body_mass_g"], ax=axes, line_kws={"color": "black"}, scatter_kws={"s": 2})
plt.plot()
fig.savefig(filename + ".jpg")

model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
print(model.fit().summary())


The variable flipper_length_mm being $\hat{\beta}_1 = 50.1533$ means that for every millimeter of flipper length extra measured on a penguin, it's likely for the response (measured body mass) to be 50.1533g heavier. A penguin with longer flippers are likely to have a heavier body mass.
The estimated 95% confidence interval for $\hat{\beta}_1$ is CI=[47.123, 53.183]. With $\alpha=0.05$, it seems the variable does affect body_mass_g positively.
The low p value shows we can be quite certain of a non-zero connection.

### TA plot

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
sns.scatterplot(
                x=result.fittedvalues,
                y=result.resid,
                ax=axes,
                s=8,
                )
plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.show()
fig.savefig("TA-plot.jpg")



It can be seen here that there seems to be 2 clusters, and something that looks like a subtle curve.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
sns.scatterplot(
                data=penguins,
                x=result.fittedvalues,
                y=result.resid,
                ax=axes,
                s=8,
                hue="sex",
                palette = {
                  "Male": "tab:blue",
                  "Female": "tab:pink",
                },
                style="species"
                )
plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
plt.show()
fig.savefig("TA-plot-species-sex.jpg")


The clusters aren't a sign of a poor model by themselves, but they do indicate there might be more to this graph. Earlier, we saw how the model doesn't hold up when filtering for the categories of sex and species, which we can tell here is likely the cause for the strange clustering.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
ax = sns.scatterplot(
  x=result.fittedvalues,
  y=result.resid,
  ax=axes,
  s=8,
)

sns.regplot(
  x=result.fittedvalues,
  y= result.resid,
  lowess=True,
  scatter=False,
  color="red",
  ax=ax
)

plt.ylabel("Residual")
plt.xlabel("Predicted value")
ax.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.show()
fig.savefig("TA-plot-lowess-line.jpg")


Here we can see there's a slight curve in the TA plot, going from a little under 500g in underestimation to about 100g in overestimation, back to around 100g in underestimation. We'll see if it disappears when including more variables later.

### QQ plot

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
sm.qqplot(result.resid, line="45", ax=axes[0] fit=True)
plt.ylabel("Quantiles in standardized residuals")
plt.xlabel("Quantiles in normal distribution")
plt.title("QQ plot")
plt.show()
fig.savefig("QQ-plot.jpg")


The QQ-plot can be seen here to seem fine, indicating a normal distribution. However, due to the issues with the TA-plot the model is still under question. The QQ-plot only shows the overall shape of the distribution, ignoring where it sits along the fitted values. Where these observations sit along the fitted values which we can observe in the TA-plot is what let us figure out the trends which make the model problematic.

## Model with species and sex
Now that we've seen the variable of flipper length might be hiding the effects of other variables, we'll create a multiple variable regression model, and drop each variable separately from the model.

In [ ]:
def genModel(vars, data=penguins):
  return smf.ols("body_mass_g ~ " + " + ".join(vars), data=data)

allVars = variables + variables2
fit = genModel(allVars).fit()
print(fit.summary())


The large condition number of 7.55e+03 is probably because species and sex affect flipper and bill sizes, causing uncertainty in the model variables.

In [ ]:

rAdjSqrdChanges = {}
for var in allVars:
    newVars = allVars.copy()
    newVars.remove(var)
    rAdjSqrdChange = fit.rsquared_adj - genModel(newVars).fit().rsquared_adj
    rAdjSqrdChanges[rAdjSqrdChange] = var
    print(f"R²_Adj change when removing {var}: {rAdjSqrdChange}")

print(f"\nThe biggest R²_Adj change is: {max(rAdjSqrdChanges.keys())} and is caused by the variable {rAdjSqrdChanges[max(rAdjSqrdChanges.keys())]}")


We can see here that species has the biggest unique contribution, and is therefore a contender for the strongest variable.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
ax = sns.scatterplot(
    data=penguins,
    x=result.fittedvalues,
    y=result.resid,
    ax=axes,
    s=8,
    hue="sex",
    palette = {
      "Male": "tab:blue",
      "Female": "tab:pink",
    },
    style="species"
)

sns.regplot(
  x=result.fittedvalues,
  y= result.resid,
  lowess=True,
  scatter=False,
  color="red",
  ax=ax
)

plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("Original TA plot")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
plt.show()


In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
ax = sns.scatterplot(
    data=penguins,
    x=fit.fittedvalues,
    y=fit.resid,
    ax=axes,
    s=8,
    hue="sex",
    palette = {
      "Male": "tab:blue",
      "Female": "tab:pink",
    },
    style="species"
)

sns.regplot(
  x=fit.fittedvalues,
  y= fit.resid,
  lowess=True,
  scatter=False,
  color="red",
  ax=ax
)

plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("New TA plot")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
plt.show()
fig.savefig("TA-plot-new.jpg")


The new model no longer underestimates males and overestimates females, and the slight curve has been reduced to under 100g underestimation on both ends of the curve, and under 100g in overestimation at the curve's extremum. This seems to be an improvement.


In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
sm.qqplot(fit.resid, line="45", ax=axes[0] fit=True)
plt.ylabel("Quantiles in standardized residuals")
plt.xlabel("Quantiles in normal distribution")
plt.title("QQ plot")
plt.show()
fig.savefig("QQ-plot-new.jpg")


The QQ plot still indicates a normal distribution, with little change from the previous model's plot.

# Sex and Species weight

Now to  our question:
Do males tend to be heavier than females?
And are gentoo penguins the heaviest, followed by chinstrap?

We've seen indications of this in several of the earlier plots, but particularly in our final model, we can see their individual contributions directly with minimal noise.

In [ ]:

sexFit = genModel(["sex","species"]).fit()
print(f"Male factor across all species: {sexFit.params['sex[T.Male]']}")
maleConf = fit.conf_int().loc["sex[T.Male]"]
print(f"Male factor confidence interval all species: [{maleConf[0]}, {maleConf[1]}]\n\n")
species = ["Gentoo", "Chinstrap", "Adelie"]
for sp in species:
    data = penguins[penguins["species"] == sp]
    newFit = genModel(["sex"], data=data).fit()
    print(f"{sp}'s male factor: {newFit.params['sex[T.Male]']}")
    maleConf = newFit.conf_int().loc["sex[T.Male]"]
    print(f"Male factor confidence interval {sp}: [{maleConf[0]}, {maleConf[1]}]\n\n")



Here we can see that with female as default, the male factor affects the weight positively, meaning that males tend to be heavier than females. With $\alpha=0.05$ we see that it has an effect across all species as well. The confidence interval shows a heavier expected weight across the whole confidence interval.

In [ ]:

newFit = genModel(["species", "sex"]).fit()
print(f"Gentoo factor: {newFit.params['species[T.Gentoo]']}")
gentooConf = newFit.conf_int().loc["species[T.Gentoo]"]
print(f"Gentoo factor confidence interval: [{gentooConf[0]}, {gentooConf[1]}]\n\n")
print(f"Chinstrap factor: {newFit.params['species[T.Chinstrap]']}")
chinstrapConf = newFit.conf_int().loc['species[T.Chinstrap]']
print(f"Chinstrap factor confidence interval: [{chinstrapConf[0]}, {chinstrapConf[1]}]\n\n")


print(newFit.t_test("species[T.Gentoo] - species[T.Chinstrap] = 0"))


Our hypothesis that gentoo penguins are the heaviest was correct, however our hypothesis that Chinstrap penguins are the second heaviest can't be verified. We can see that adelie penguins are the default in our model, and that gentoo penguins and chinstrap penguins have a higher expected weight. This would mean that the order of weights is gentoo, chinstrap, adelie, like our hypothesis. However, the confidence interval for chinstrap's weight with $\alpha=0.05$ includes a negative effect on the lower end, which means we can't ascertain the significance of the chinstrap species variable. This means our hypothesis is only partially demonstrable.
A direct comparison of gentoo penguins with chinstrap penguins show gentoo penguins are also heavier than chinstrap penguins on both ends of the confidence interval.

# Conclusion (for a full conclusion and reflection, read the document)
Our hypothesis that flipper length is the strongest predictor depends on the definition of predictor strength. By itself it has the best R² value, however its unique contribution to R² value in the complete model is beat by species in first and sex in second. This is because flipper length works as a stand-in for both species and sex, over-inflating its importance outside of a full model.

Our hypothesis that males are heavier than females was supported for the penguins generally, and within specific penguin species. Both ends of the confidence interval was positive, meaning males tend to be heavier than females for all plausible expected values with $\alpha=0.05$.

Lastly, our hypothesis for gentoo being the heaviest followed by chinstrap penguins is also not fully supported. Gentoo penguins being plausibly the heaviest has been well demonstrated, with the lightest plausible expected weight being higher than the adelie expected weight and the highest plausible chinstrap expected weight. But, the weight of chinstrap penguins compared to adelie penguins includes plausible differences from negative to positive values. Meaning chinstrap penguins could plausibly have a less heavy or equally heavy expected value as adelie penguins.

# Reflection
How could our hypothesis be properly validated/invalidated?
If we had more measurements for chinstrap penguins, the uncertainty of the confidence intervals would be reduced, and with a big enough sample we might be able to find whether chinstrap penguins are heavier than adelie penguins or not.